# Bead volume via disk integration - **batch version (many images at once)**

Side-view silhouette -> r(z) per pixel row -> `V = sum(pi * r_i^2 * dz)`

Same physics and same maths as the single-image notebook, but now:

- you select **any number of images** in one go (multi-select dialog, a folder, or a typed list)
- the ROI is checked for you: if the crop cuts the bead off, the run says so loudly instead of
  quietly returning a set of identical volumes
- every image is processed with the same settings and collected into **one summary table**
  (one row per image: height, base radius, volume by disk integration, volume of the equivalent
  spherical cap, contact angle, ...)
- the per-row `r_i` tables for every image are kept too, in both cropped and full-image coordinates
- everything is written to a single Excel workbook (`bead_volumes.xlsx`) with a `summary` sheet
  plus one sheet per image, and to CSVs
- a set of plots for a shrinkage experiment: QC montage, overlaid `r(z)` profiles, normalised
  shape overlay, volume bar chart, **volume shrinkage / linear strain vs time**, and a
  disk-vs-spherical-cap agreement check

Two small robustness changes vs the single-image notebook:

1. **thresholding is done on the cropped ROI, not on the full frame.** Otsu picks its threshold
   from the pixels it is given, so cropping first stops the microscope's black surround and the
   on-screen text from dragging the threshold around. Each image therefore gets a threshold
   suited to itself.
2. the mask is **cleaned** (small closing + hole filling) before the largest blob is taken, so a
   glare spot inside the bead does not punch a hole in the silhouette and shrink the radius.
3. the base radius `a` defaults to the median of the widest 5% of rows rather than the single
   widest row, so a one-row flare at the contact line cannot inflate it (`A_METHOD = "max"`
   restores the original behaviour).

Run every cell **in order, top to bottom**.

In [ ]:
import os, re, glob, math, sys
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
!{sys.executable} -m pip install openpyxl
%matplotlib inline

## Step 0 - pick your images

Three ways to hand the notebook a set of images; the first one that yields files wins:

- **A** - type the paths into `IMAGE_PATHS`
- **B** - point `IMAGE_FOLDER` at a folder and every image in it is used
- **C** - leave both empty and a multi-select dialog opens (Ctrl-click / Shift-click to select
  many files at once)

Files are sorted *naturally*, so `t2` comes before `t10`. That order is the order used in every
table and plot below, and for shrinkage it is assumed to be the time order of the experiment.

In [ ]:
IMAGE_PATHS = [                 # A: paste paths here, e.g. r"C:\data\bead_t0.jpg",
]
IMAGE_FOLDER = None             # B: e.g. r"C:\data\bead_run_1"  (all images in the folder)
USE_FILE_DIALOG = True          # C: fall back to a multi-select dialog

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def natural_key(path):
    """Sort helper so bead_t2 comes before bead_t10."""
    name = os.path.basename(path).lower()
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", name)]

paths = [p for p in IMAGE_PATHS if p]

if not paths and IMAGE_FOLDER:
    paths = [p for p in glob.glob(os.path.join(IMAGE_FOLDER, "*"))
             if p.lower().endswith(EXTS)]

if not paths and USE_FILE_DIALOG:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    paths = list(filedialog.askopenfilenames(
        title="Select ALL bead side-view images (Ctrl-click / Shift-click for multiple)",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.tif *.tiff"), ("All files", "*.*")],
    ))
    root.destroy()

IMAGE_PATHS = sorted(dict.fromkeys(paths), key=natural_key)

if not IMAGE_PATHS:
    raise RuntimeError("No images selected - set IMAGE_PATHS or IMAGE_FOLDER, or re-run and pick files.")

print(f"{len(IMAGE_PATHS)} image(s) selected, in processing order:")
for i, p in enumerate(IMAGE_PATHS):
    print(f"  [{i}] {os.path.basename(p)}")

If the dialog never appears, check the taskbar (it can open behind the browser). If it is blocked
entirely, use option A or B in the cell above - e.g. `IMAGE_FOLDER = r"C:\path\to\folder"`.

## Parameters

`DEFAULT_ROI` applies to every image. If one image needs a different crop (the bead drifted, the
stage moved), add an entry to `ROI_OVERRIDES` keyed by the **file name**; only that image uses it.

Nothing else needs editing per image set: the cropped-to-full-image conversion in the tables
(`Y_full = y0 + y_crop`, `x_left_full = x0 + x_crop`) is taken from whichever ROI each image
actually used, override included, so it follows the crop automatically. **`SCALE_PX_PER_UM` is the
one number that must be re-derived** whenever the magnification, the working distance or the image
size changes - a scale calibrated on a full-size frame is wrong by exactly the resize factor if you
then analyse resized copies.

`INVERT`, `MANUAL_THRESH` and `SCALE_PX_PER_UM` work exactly as in the single-image notebook.
Because each image is thresholded inside its own ROI, one `INVERT` setting normally covers a whole
run shot under the same lighting.

In [ ]:
SCALE_PX_PER_UM = None       # pixels per micron from the REFLECTED-path calibration.
                             # Leave as None to work in raw pixels (volumes then come out in px^3).
INVERT = False               # True if bead is dark-on-light; False if light-on-dark.
MANUAL_THRESH = None         # 0-255 to override Otsu auto-threshold, or None
THRESH_OFFSET = 0            # shift the automatic Otsu level by this much. Negative includes more
                             # of a dim, shadowed bead base; ignored when MANUAL_THRESH is set.

# Which connected blob is the bead. "best" scores every blob on how bead-like it is (a dome
# resting on the bottom edge) and takes the winner - use this when something else in the frame,
# such as an out-of-focus bright object beside the bead, is physically bigger than the bead.
# "largest" takes the biggest blob, as the original notebook did.
BLOB_CHOICE = "best"

# How the bead is separated from everything else. Otsu assumes the crop is roughly two-toned
# (bead vs background); if that is not true for your images, Step 0b will tell you which of these
# does work:
#   "otsu"     - global Otsu (or MANUAL_THRESH) on the crop            <- original behaviour
#   "adaptive" - local threshold, for uneven lighting / a bright mat
#   "edges"    - Canny outline, closed and filled: works when the bead and its background are
#                nearly the same brightness but the rim is still visible
SEGMENT_METHOD = "otsu"

# Region of interest = (x0, x1, y0, y1) in the FULL image's pixel coordinates.
# Cropped BEFORE thresholding, so overlay text / microscope frame never become "foreground".
# Set it tight enough to exclude the text and the empty background, and to cut the image off at
# the silicon mat surface (the bottom edge y1 acts as the contact line / baseline).
DEFAULT_ROI = (800, 1600, 700, 1075)

# Per-image exceptions, keyed by file name:
#   ROI_OVERRIDES = {"bead_t10.jpg": (820, 1620, 690, 1070)}
ROI_OVERRIDES = {}

# Mask cleanup before the largest blob is picked.
CLOSE_PX   = 5               # morphological closing kernel in px (0 = off) - seals thin gaps
FILL_HOLES = True            # fill glare holes inside the silhouette

# Base radius a: "robust" = median of the widest 5% of rows (ignores a one-row flare at the
# contact line), "max" = the single widest row, as in the original notebook.
A_METHOD = "robust"

MONTAGE_MAX = 12             # at most this many panels in the QC montage, sampled evenly

# Optional time axis. If your file names carry the time point, give a regex with ONE capture
# group of digits, e.g. r"_t(\d+)min"  ->  bead_t30min.jpg gives t = 30.
# Leave as None to just use the image order (0, 1, 2, ...).
TIME_REGEX = None
TIME_UNIT  = "min"

# Row of the mat surface (the contact line) in FULL-image coordinates, if you know it - read it
# off the original photo, or from a length measurement drawn in the microscope software.
# With it set, the notebook reports how far short of the mat each mask stops, and adds V_base:
# the volume integrated to the widest row plus a cylinder of radius a down to the mat. That is a
# salvage estimate for images whose base is lost to shadow; the real fix is a threshold (or a
# backlight) that lets the silhouette reach the mat on its own.
BASELINE_Y_FULL = None

# Which image is the un-shrunk reference for the strain calculation (0 = the first one).
REFERENCE_INDEX = 0

OUT_PREFIX = "bead_volumes"  # output files: bead_volumes.xlsx / _summary.csv / _profiles.csv

## Helper functions

`analyze_bead()` is the single-image pipeline from the original notebook wrapped into one function
so it can be applied to every image: crop -> threshold -> clean -> largest blob -> apex -> r(z)
profile -> volume. It returns the profile table, the derived numbers and the images needed for QC
plots, and it never raises for a single bad image - the failure is recorded and the batch goes on.

In [ ]:
def threshold_crop(img, invert=False, manual_thresh=None, method="otsu"):
    """Foreground mask of an already-cropped grayscale image. See SEGMENT_METHOD."""
    blur = cv2.GaussianBlur(img, (5, 5), 0)

    if method == "adaptive":
        # block size ~ a third of the crop, forced odd; C<0 keeps the brighter side as foreground
        bs = max(11, (min(img.shape) // 3) | 1)
        mask = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, bs, -5)
    elif method == "edges":
        v = float(np.median(blur))
        edges = cv2.Canny(blur, int(max(0, 0.66 * v)), int(min(255, 1.33 * v)))
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, k)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        mask = np.zeros_like(edges)
        cv2.drawContours(mask, contours, -1, 255, thickness=cv2.FILLED)
    elif manual_thresh is not None:
        _, mask = cv2.threshold(blur, manual_thresh, 255, cv2.THRESH_BINARY)
    else:
        level, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if THRESH_OFFSET:
            _, mask = cv2.threshold(blur, max(0, min(255, level + THRESH_OFFSET)), 255,
                                    cv2.THRESH_BINARY)

    if invert:
        mask = cv2.bitwise_not(mask)
    return mask


def mask_quality(mask):
    """How much does this mask look like a bead sitting on the bottom edge of the crop?

    A bead silhouette: sits on the mat (touches the bottom), is clear of the top and the sides,
    takes up a sensible fraction of the crop, and widens steadily from apex to base. Anything
    that is really the background, the mat or a text overlay fails at least one of those.
    """
    h, w = mask.shape
    ys, xs = np.where(mask > 0)
    if ys.size == 0:
        return {"score": -1.0, "area_frac": 0.0, "touches": "nothing", "dome": 0.0}

    area_frac = ys.size / float(h * w)
    t_top, t_bot = bool((ys == 0).any()), bool((ys == h - 1).any())
    t_l, t_r = bool((xs == 0).any()), bool((xs == w - 1).any())

    rows, widths, _, _ = bead_profile(mask)
    dome = float(np.mean(np.diff(widths) >= 0)) if widths.size > 2 else 0.0   # widens downwards

    # Solidity separates a solid silhouette from a rim, a ragged fragment or a speckled region:
    # a bead fills its own convex hull almost completely.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(contours, key=cv2.contourArea)
    hull_area = cv2.contourArea(cv2.convexHull(c))
    solidity = float(cv2.contourArea(c) / hull_area) if hull_area > 0 else 0.0

    score = 1.5 * dome                       # widens from apex to base
    score += 1.5 * solidity - 0.75           # solid, not a rim or a scatter of specks
    score += 0.5 * min(1.0, area_frac / 0.10)   # a real bead is not a speck
    score += 0.5 if t_bot else -0.5          # a bead rests on the mat
    score -= 1.0 * (t_top + t_l + t_r)       # anything hitting the top or a side is cut off
    if not (0.03 <= area_frac <= 0.75):      # far too small (noise) or far too big (background)
        score -= 1.5
    touches = ",".join([n for n, f in (("top", t_top), ("bottom", t_bot),
                                       ("left", t_l), ("right", t_r)) if f]) or "none"
    return {"score": float(score), "area_frac": float(area_frac), "solidity": solidity,
            "touches": touches, "dome": dome}


def clean_mask(mask, close_px=5, fill_holes=True):
    """Seal thin gaps and fill interior holes (glare) so the silhouette stays solid."""
    if close_px and close_px > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(close_px), int(close_px)))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    if fill_holes:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filled = np.zeros_like(mask)
        cv2.drawContours(filled, contours, -1, 255, thickness=cv2.FILLED)
        mask = filled
    return mask


def largest_component(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        raise RuntimeError("No foreground blob found - check threshold/invert flag")
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return np.uint8(labels == idx) * 255


def pick_blob(mask, mode="best", max_candidates=6):
    """Choose the bead's blob. "best" ranks blobs by how much each looks like a bead.

    The largest bright blob is not always the bead: an out-of-focus lamp, a holder or a bright
    mat can be physically bigger. Scoring each candidate on shape and position finds the bead
    anyway, as long as it is a separate blob from whatever else is bright.
    """
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        raise RuntimeError("No foreground blob found - check threshold/invert flag")
    areas = stats[1:, cv2.CC_STAT_AREA]
    order = np.argsort(-areas) + 1
    if mode != "best":
        return np.uint8(labels == order[0]) * 255

    floor = max(0.05 * areas.max(), 50)          # ignore specks
    cands = [i for i in order[:max_candidates] if stats[i, cv2.CC_STAT_AREA] >= floor]
    scored = []
    for i in cands:
        blob = np.uint8(labels == i) * 255
        scored.append((mask_quality(blob)["score"], stats[i, cv2.CC_STAT_AREA], i))
    best_score = max(sc for sc, _, _ in scored)
    # among blobs that score about as well, keep the biggest
    i = max((a, i) for sc, a, i in scored if sc >= best_score - 0.25)[1]
    return np.uint8(labels == i) * 255


def bead_profile(mask, apex_y=None):
    """Per-row half-width (radius) of the mask, top to bottom, in the mask's own coordinates."""
    ys, xs = np.where(mask > 0)
    y_min, y_max = ys.min(), ys.max()
    if apex_y is not None:
        y_min = max(y_min, int(round(apex_y)))
    widths, rows, lefts, rights = [], [], [], []
    for y in range(y_min, y_max + 1):
        row_xs = xs[ys == y]
        if row_xs.size == 0:
            continue
        xl, xr = int(row_xs.min()), int(row_xs.max())
        rows.append(y); lefts.append(xl); rights.append(xr)
        widths.append(xr - xl + 1)
    return (np.array(rows), np.array(widths, dtype=float),
            np.array(lefts), np.array(rights))


def base_radius(widths_px, method="robust"):
    """Half-width of the bead's base. 'robust' ignores a one-row flare at the contact line."""
    if method != "robust" or widths_px.size < 20:
        return float(widths_px.max()) / 2.0
    thr = np.percentile(widths_px, 95)
    return float(np.median(widths_px[widths_px >= thr])) / 2.0


def roi_for(path, default_roi=None, overrides=None):
    return (overrides or {}).get(os.path.basename(path), default_roi)


def time_for(path, index, regex=None):
    """Time point from the file name if TIME_REGEX matches, else the image index."""
    if regex:
        m = re.search(regex, os.path.basename(path))
        if m:
            return float(m.group(1))
    return float(index)


def analyze_bead(path, roi=None, invert=False, manual_thresh=None, scale=None,
                 close_px=5, fill_holes=True, apex_override=None, a_method="robust",
                 method="otsu", baseline_y=None):
    """Full single-image pipeline. Returns a dict with the profile table, the numbers and QC images."""
    img_full = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img_full is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    H, W = img_full.shape

    x0, x1, y0, y1 = roi if roi else (0, W, 0, H)
    x0, x1 = max(0, int(x0)), min(W, int(x1))
    y0, y1 = max(0, int(y0)), min(H, int(y1))
    if x1 - x0 < 2 or y1 - y0 < 2:
        raise ValueError(f"ROI {roi} does not overlap the image ({W}x{H})")

    img  = img_full[y0:y1, x0:x1]
    mask = threshold_crop(img, invert=invert, manual_thresh=manual_thresh, method=method)
    mask = clean_mask(mask, close_px=close_px, fill_holes=fill_holes)
    mask = pick_blob(mask, BLOB_CHOICE)

    ys, xs = np.where(mask > 0)

    # Does the silhouette run into the edge of the crop? Touching the BOTTOM is expected - that
    # edge is the contact line. Touching the top or a side means the ROI is cutting the bead off,
    # and every number below would then be a property of the crop, not of the bead.
    mh, mw = mask.shape
    clip_top   = bool((ys == 0).any())
    clip_bottom = bool((ys == mh - 1).any())
    clip_left  = bool((xs == 0).any())
    clip_right = bool((xs == mw - 1).any())
    clipped = ",".join([n for n, f in (("top", clip_top), ("left", clip_left),
                                       ("right", clip_right)) if f])

    apex_y = int(ys.min()) if apex_override is None else int(apex_override[1])
    apex_x = float(xs[ys == apex_y].mean()) if apex_override is None else float(apex_override[0])

    rows, widths_px, lefts, rights = bead_profile(mask, apex_y=apex_y)
    if rows.size == 0:
        raise RuntimeError("Empty profile - mask has no rows below the apex")

    r_px = widths_px / 2.0
    z_px = rows - rows.min()

    prof = pd.DataFrame({
        "z_px":          z_px.astype(int),
        "y_crop":        rows.astype(int),
        "Y_full":        (y0 + rows).astype(int),
        "x_left_full":   x0 + lefts,
        "x_right_full":  x0 + rights,
        "x_center_full": x0 + (lefts + rights) / 2.0,
        "width_px":      widths_px,
        "r_i_px":        r_px,
    })

    # --- volume: V = sum(pi * r_i^2 * dz), dz = 1 px ---------------------------------
    h_px     = float(len(rows))                 # height = number of rows, dz = 1 px
    a_px     = base_radius(widths_px, a_method)        # base radius (see A_METHOD)
    V_px3    = float(np.pi * np.sum(r_px ** 2))

    # Everything below the widest row is the least trustworthy part of the silhouette: it is
    # where a shadow at the mat, or the threshold, eats into the base. V_trunc integrates only
    # down to the widest row, i.e. treats that row as the contact line. For a bead that meets
    # the mat at 90 degrees or less the two agree; where they disagree, the base is in doubt.
    i_widest = int(np.argmax(widths_px))
    V_trunc_px3 = float(np.pi * np.sum(r_px[:i_widest + 1] ** 2))

    # With the mat's row known, how far short of it does this mask stop, and what would the bead
    # hold if it kept the width of its widest row the rest of the way down?
    rows_short = np.nan
    V_base_px3 = np.nan
    if baseline_y is not None:
        rows_short = float(baseline_y - (y0 + rows[-1]))
        fill_rows = max(0.0, float(baseline_y - (y0 + rows[i_widest])))
        V_base_px3 = V_trunc_px3 + float(np.pi * r_px[i_widest] ** 2 * fill_rows)
    V_cap_px3 = float((np.pi * h_px / 6.0) * (3 * a_px ** 2 + h_px ** 2))
    theta_deg = float(2 * math.degrees(math.atan2(h_px, a_px))) if a_px > 0 else np.nan

    # A sessile bead is widest where it meets the mat. If the silhouette has narrowed well below
    # its widest row before the crop ends, the threshold is losing a dim, shadowed base.
    base_taper = float(widths_px[-1] / widths_px.max()) if widths_px.max() else np.nan

    m = {
        "image":        os.path.basename(path),
        "path":         path,
        "roi":          (x0, x1, y0, y1),
        "frame_w":      int(W),
        "frame_h":      int(H),
        "apex_x_crop":  apex_x,
        "apex_y_crop":  apex_y,
        "apex_X_full":  x0 + apex_x,
        "apex_Y_full":  y0 + apex_y,
        "base_taper":         base_taper,       # bottom row width / widest row width
        "clipped":            clipped,          # "" when the ROI encloses the bead properly
        "touches_roi_bottom": clip_bottom,      # normal: the ROI's bottom edge is the mat
        "n_rows":       int(len(rows)),
        "area_px":      int(np.count_nonzero(mask)),
        "h_px":         h_px,
        "a_px":         a_px,
        "width_max_px": float(widths_px.max()),
        "V_disk_px3":   V_px3,
        "V_trunc_px3":  V_trunc_px3,
        "V_base_px3":   V_base_px3,
        "rows_short":   rows_short,
        "widest_frac":  float(i_widest / max(1, len(rows) - 1)),   # 1.0 = widest row is the last
        "Y_widest_full": int(y0 + rows[i_widest]),                 # fixed camera -> should not move
        "Y_bottom_full": int(y0 + rows[-1]),
        "V_cap_px3":    V_cap_px3,
        "cap_diff_pct": 100.0 * (V_px3 - V_cap_px3) / V_cap_px3 if V_cap_px3 else np.nan,
        "contact_angle_deg": theta_deg,
        "aspect_h_over_a":   h_px / a_px if a_px else np.nan,
    }

    if scale:
        prof["z_um"]   = prof["z_px"]   / scale
        prof["r_i_um"] = prof["r_i_px"] / scale
        m["h_um"]      = h_px / scale
        m["a_um"]      = a_px / scale
        m["V_disk_um3"] = V_px3 / scale ** 3
        m["V_trunc_um3"] = V_trunc_px3 / scale ** 3
        m["V_base_um3"]  = V_base_px3 / scale ** 3
        m["V_cap_um3"]  = V_cap_px3 / scale ** 3
        m["V_disk_mm3"] = m["V_disk_um3"] / 1e9

    return {"metrics": m, "profile": prof, "img": img, "mask": mask,
            "apex": (apex_x, apex_y)}

## Step 0b - the code cannot find the bead? Start here

Run this on one image. It tries every segmentation route on the same crop, scores each one on how
much the result looks like **a bead sitting on the mat** - widens from apex to base, touches the
bottom edge, clear of the top and sides, sensible size - and prints what to set.

The two extra panels are the ones that usually explain the failure:

- **the crop itself**: if the bead is not clearly inside it, no threshold will save you - fix the
  ROI first (Step 0c)
- **the histogram**: Otsu only works when the crop is roughly two-toned, i.e. the histogram has two
  humps with a valley between them. One broad hump means there is no single grey level that
  separates bead from background - use `"adaptive"` or `"edges"`, or read a level off the histogram
  by eye and set `MANUAL_THRESH`

A very common cause on a mat: the crop includes so much **mat** that the mat, not the bead, is the
biggest bright object, so `largest_component` returns the mat. The giveaway is a top-scoring mask
that touches the left *and* right edges. Pull the ROI's bottom edge up to the contact line.

In [ ]:
DIAG_INDEX   = 0        # which image to diagnose (index from the Step 0 listing)
DIAG_USE_ROI = True     # True: diagnose inside DEFAULT_ROI. False: the whole frame.


def diagnose_segmentation(path, roi=None, manual_thresh=MANUAL_THRESH, show=True):
    """Try every segmentation route on one image and rank the results."""
    img_full = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img_full is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    H, W = img_full.shape
    x0, x1, y0, y1 = roi if roi else (0, W, 0, H)
    x0, x1 = max(0, int(x0)), min(W, int(x1))
    y0, y1 = max(0, int(y0)), min(H, int(y1))
    img = img_full[y0:y1, x0:x1]

    cands, rows = {}, []
    for method in ("otsu", "adaptive", "edges"):
        for inv in (False, True):
            if method == "edges" and inv:
                continue                      # a filled outline has no meaningful inverse
            name = method + (" + INVERT" if inv else "")
            try:
                m = pick_blob(clean_mask(
                    threshold_crop(img, inv, manual_thresh, method), CLOSE_PX, FILL_HOLES),
                    BLOB_CHOICE)
            except Exception as e:
                rows.append({"setting": name, "score": -99.0, "note": f"failed: {e}"})
                continue
            qy = mask_quality(m)
            cands[name] = m
            rows.append({"setting": name, "score": qy["score"], "widens_down": qy["dome"],
                         "area_frac": qy["area_frac"], "touches": qy["touches"],
                         "SEGMENT_METHOD": method, "INVERT": inv})

    table = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

    if show:
        blur = cv2.GaussianBlur(img, (5, 5), 0)
        otsu_level, _ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        ncols = 3
        nrows = math.ceil((2 + len(cands)) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.6 * nrows), squeeze=False)
        flat = list(axes.ravel())
        for ax in flat:
            ax.axis("off")

        flat[0].imshow(img, cmap="gray")
        flat[0].set_title("the crop being segmented", fontsize=10)

        hax = flat[1]
        hax.axis("on")
        hax.hist(img.ravel(), bins=64, color="#2a78d6")
        hax.axvline(otsu_level, color="#e34948", linewidth=2)
        hax.set_title(f"grey-level histogram (Otsu = {otsu_level:.0f})", fontsize=10)
        hax.set_yticks([])
        hax.tick_params(labelsize=8)
        for side in ("top", "right", "left"):
            hax.spines[side].set_visible(False)

        order = [r for r in table["setting"] if r in cands]
        for ax, name in zip(flat[2:], order):
            ax.imshow(img, cmap="gray")
            ax.contour(cands[name], levels=[127], colors="#e34948", linewidths=1.2)
            sc = float(table.loc[table["setting"] == name, "score"].iloc[0])
            ax.set_title(f"{name}   score {sc:+.2f}", fontsize=10)
        fig.suptitle(os.path.basename(path), fontsize=11)
        fig.tight_layout()
        plt.show()

    return table


diag = diagnose_segmentation(IMAGE_PATHS[DIAG_INDEX], DEFAULT_ROI if DIAG_USE_ROI else None)
print(diag.to_string(index=False))

best = diag.iloc[0]
if best["score"] < 1.0:
    print("\nNone of these look like a bead on a mat.")
    print("  - is the bead actually inside the crop? (first panel)")
    print("  - does the histogram have two humps? if not, try MANUAL_THRESH by eye,")
    print("    or pull the ROI in so the crop holds little more than the bead")
    print("  - if the mat is the brightest thing in the crop, raise the ROI's bottom edge")
else:
    print(f"\nBest: SEGMENT_METHOD = {best['SEGMENT_METHOD']!r}   INVERT = {best['INVERT']}")
    print("Set those in the parameters cell, re-run it, then carry on to Step 0c / Step 1.")

## Step 0c - not sure what ROI to use? Get a suggestion

**Do not keep the `DEFAULT_ROI` from someone else's images.** If the crop cuts the bead off, the
apex the notebook finds is the edge of the crop, the height is the crop's height, and every volume
comes out the same - a wrong answer that looks like a clean measurement.

This cell thresholds the *whole* frame, takes the largest blob, and pads its bounding box on the
top and sides to suggest an ROI. The suggestion is **drawn, not applied**: check that the green box
surrounds the bead with background above the apex, then copy the printed tuple into `DEFAULT_ROI`
above, re-run the parameters cell, and carry on.

The bottom edge is left on the blob's own bottom row - that is the contact line with the mat. If
the bead's blob merges with a bright mat, or the microscope's text overlay is the biggest bright
thing in the frame, the suggestion will be wrong and you should read the box off the plot by hand.

In [ ]:
def suggest_roi(path, invert=INVERT, manual_thresh=MANUAL_THRESH, margin=0.18, show=True,
                method=None):
    """Propose an ROI from the largest blob in the FULL frame. Check it by eye before using it."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    H, W = img.shape
    m = pick_blob(clean_mask(
        threshold_crop(img, invert, manual_thresh, method or SEGMENT_METHOD),
        CLOSE_PX, FILL_HOLES), BLOB_CHOICE)
    ys, xs = np.where(m > 0)
    bx0, bx1 = int(xs.min()), int(xs.max()) + 1
    by0, by1 = int(ys.min()), int(ys.max()) + 1
    mx, my = int(margin * (bx1 - bx0)), int(margin * (by1 - by0))
    roi = (max(0, bx0 - mx), min(W, bx1 + mx), max(0, by0 - my), by1)   # no padding below

    if show:
        x0, x1, y0, y1 = roi
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(img, cmap="gray")
        ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False,
                                   color="#e34948", linewidth=1.2, linestyle="--"))
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                   color="#1baf7a", linewidth=2))
        ax.set_title(f"{os.path.basename(path)}\nred dashed = detected blob, green = suggested ROI "
                     f"{roi}", fontsize=10)
        ax.axis("off")
        plt.tight_layout(); plt.show()
    return roi


probe = list(dict.fromkeys([IMAGE_PATHS[0],
                            IMAGE_PATHS[len(IMAGE_PATHS) // 2],
                            IMAGE_PATHS[-1]]))
rois = [suggest_roi(p) for p in probe]
union = (min(r[0] for r in rois), max(r[1] for r in rois),
         min(r[2] for r in rois), max(r[3] for r in rois))

print("Suggested ROI covering the first / middle / last image:")
print(f"    DEFAULT_ROI = {union}")
print("Paste that into the parameters cell if the green boxes look right, then re-run it.")

## Step 1 - run the batch

Every image goes through the same pipeline. If one image fails (bad ROI, nothing found at the
chosen threshold) it is reported and skipped, and the rest still run.

In [ ]:
results, failures = [], []

for i, p in enumerate(IMAGE_PATHS):
    try:
        res = analyze_bead(
            p,
            roi=roi_for(p, DEFAULT_ROI, ROI_OVERRIDES),
            invert=INVERT,
            manual_thresh=MANUAL_THRESH,
            scale=SCALE_PX_PER_UM,
            close_px=CLOSE_PX,
            fill_holes=FILL_HOLES,
            a_method=A_METHOD,
            method=SEGMENT_METHOD,
            baseline_y=BASELINE_Y_FULL,
        )
        res["metrics"]["index"] = i
        res["metrics"]["label"] = os.path.splitext(os.path.basename(p))[0]
        res["metrics"]["time"]  = time_for(p, i, TIME_REGEX)
        results.append(res)
        m = res["metrics"]
        flag = f"  <-- CLIPPED at {m['clipped']}" if m["clipped"] else ""
        print(f"[{i}] {m['image']:<32s} h={m['h_px']:6.1f} px  a={m['a_px']:6.1f} px  "
              f"V={m['V_disk_px3']:.4e} px^3{flag}")
    except Exception as e:
        failures.append((p, repr(e)))
        print(f"[{i}] {os.path.basename(p):<32s} FAILED: {e}")

print(f"\n{len(results)} of {len(IMAGE_PATHS)} image(s) analysed successfully.")
if failures:
    print("Failed images (fix the ROI / INVERT / threshold for these and re-run):")
    for p, e in failures:
        print("  -", os.path.basename(p), "->", e)
if not results:
    raise RuntimeError("Nothing analysed - check DEFAULT_ROI and INVERT.")

# ---- sanity checks on the batch as a whole -------------------------------------------------
clipped = [r["metrics"] for r in results if r["metrics"]["clipped"]]
heights = np.array([r["metrics"]["h_px"] for r in results])

if clipped:
    print("\n" + "=" * 78)
    print(f"WARNING: {len(clipped)} of {len(results)} silhouette(s) run into the edge of the ROI.")
    print("The crop is cutting the bead off, so h / a / V below describe the CROP, not the bead.")
    for m in clipped[:10]:
        print(f"  - {m['image']}: touches {m['clipped']}  (roi={m['roi']})")
    if len(clipped) > 10:
        print(f"  ... and {len(clipped) - 10} more")
    print("Fix: widen DEFAULT_ROI (Step 0b suggests one) so there is background above the apex")
    print("and on both sides, with only the bottom edge sitting on the mat, then re-run.")
    print("=" * 78)

tapered = [r["metrics"] for r in results
           if not r["metrics"]["touches_roi_bottom"] and r["metrics"]["base_taper"] < 0.7]
if tapered:
    print(f"\nNOTE: {len(tapered)} of {len(results)} silhouette(s) narrow to less than 70% of their "
          "widest row before the crop ends.")
    print("A bead resting on a mat is widest where it touches it, so the threshold is probably")
    print("losing a dim or shadowed base and the volume is an underestimate. Lower the level:")
    print("  THRESH_OFFSET = -20   (or set MANUAL_THRESH by eye from the Step 0b histogram)")
    print("and check in the Step 1b montage that the outline reaches the contact line.")

if BASELINE_Y_FULL:
    short = np.array([r["metrics"]["rows_short"] for r in results])
    if np.nanmedian(short) > 5:
        print(f"\nNOTE: the silhouettes stop a median of {np.nanmedian(short):.0f} px short of the "
              f"mat at Y = {BASELINE_Y_FULL} (worst {np.nanmax(short):.0f} px).")
        print("Those missing rows are the widest part of the bead, so V_disk is an underestimate")
        print("by more than the row count suggests. V_base fills them in at constant radius; the")
        print("better answer is a lower threshold (THRESH_OFFSET) so the mask reaches the mat.")

ywide = np.array([r["metrics"]["Y_widest_full"] for r in results])
if len(ywide) > 2 and np.ptp(ywide) > 0.02 * (results[0]["metrics"]["roi"][3]
                                              - results[0]["metrics"]["roi"][2]):
    print(f"\nNOTE: the widest row moves {np.ptp(ywide)} px across the run "
          f"(Y = {ywide.min()} to {ywide.max()} in the original image).")
    print("With a fixed camera the mat cannot move, so either the bead is genuinely changing")
    print("shape at its base (a bulging bead relaxing onto the mat), or the base is being lost")
    print("to shadow by a different amount in each image. Compare V_disk with V_trunc: if they")
    print("disagree by more than a few percent, the base is in doubt and so is the volume.")

if len(heights) > 2 and np.ptp(heights) == 0:
    print("\nWARNING: every image returned exactly the same height "
          f"({heights[0]:.0f} px). A real bead never does that - the apex is almost certainly")
    print("the top edge of the crop. Check the ROI before reading anything into the volumes.")

## Step 1b - QC montage: **look at this before trusting any number**

If any silhouette was flagged as clipped, those images are shown first - they are the ones to fix.
Otherwise, for a long run, `MONTAGE_MAX` panels are sampled evenly across the set.

Each panel is one image's crop with the detected silhouette outlined in red and the auto-detected
apex marked with a green `+`. Check that:

- the red outline hugs the bead, with no text, frame edge or reflection caught in it
- the green `+` sits on the real tip, not on a glare spot
- the bottom of the outline sits at the mat surface (the ROI's bottom edge), not above or below it
- there is **background above the apex and on both sides** - if the outline runs off the top or the
  side of a panel, the crop is cutting the bead and that image's numbers are the crop's, not the
  bead's

If one panel is wrong, fix that image with a `ROI_OVERRIDES` entry (or flip `INVERT`) and re-run
from Step 1.

In [ ]:
def montage(results, what="overlay", ncols=3, panel=3.4, max_panels=None):
    max_panels = max_panels or MONTAGE_MAX
    flagged = [r for r in results if r["metrics"]["clipped"]]
    if flagged:
        results = flagged[:max_panels]
        print(f"showing {len(results)} of the {len(flagged)} image(s) whose silhouette hits the "
              f"edge of the ROI - fix these first")
    elif len(results) > max_panels:
        keep = np.unique(np.linspace(0, len(results) - 1, max_panels).round().astype(int))
        results = [results[i] for i in keep]
        print(f"{len(keep)} of the set shown, sampled evenly across the run "
              f"(raise MONTAGE_MAX to see more)")
    n = len(results)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel * ncols, panel * nrows * 0.95),
                             squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, res in zip(axes.ravel(), results):
        m = res["metrics"]
        if what == "mask":
            ax.imshow(res["mask"], cmap="gray")
        else:
            ax.imshow(res["img"], cmap="gray")
            ax.contour(res["mask"], levels=[127], colors="#e34948", linewidths=1.2)
            ax.plot(*res["apex"], "+", color="#1baf7a", markersize=14, markeredgewidth=2.5)
        ax.set_title(f"[{m['index']}] {m['label']}", fontsize=9)
    fig.suptitle("Detected silhouette (red) and apex (green +)" if what != "mask"
                 else "Binary masks used for the volume integration", fontsize=11)
    fig.tight_layout()
    plt.show()

montage(results, what="overlay")
montage(results, what="mask")

## Step 2 - the summary table (one row per image)

`clipped` is the one to read first: anything other than an empty string means the silhouette ran
into the top or a side of the ROI, and that row's `h`, `a` and volumes are properties of the crop.

`V_disk` is the disk-integration volume, `V_cap` the volume of the spherical cap with the same
height and base radius, and `cap_diff_pct` how far the real bead departs from that ideal cap -
a large value means the bead is not cap-shaped (slumped, pinned, or a bad mask).

With `SCALE_PX_PER_UM = None` the volumes are in **px^3**; set the scale and the µm/mm^3 columns
appear automatically.

In [ ]:
summary = pd.DataFrame([r["metrics"] for r in results])

head_cols = ["index", "label", "image", "time", "clipped", "base_taper", "n_rows", "h_px", "a_px"]
tail_cols = ["width_max_px", "rows_short", "V_disk_px3", "V_trunc_px3", "V_base_px3",
             "V_cap_px3", "cap_diff_pct",
             "contact_angle_deg", "aspect_h_over_a", "area_px", "widest_frac",
             "touches_roi_bottom", "apex_Y_full", "Y_widest_full", "Y_bottom_full",
             "apex_X_full", "roi", "frame_w", "frame_h", "path"]
um_cols = [c for c in ["h_um", "a_um", "V_disk_um3", "V_trunc_um3", "V_base_um3", "V_cap_um3",
                       "V_disk_mm3"] if c in summary]
summary = summary[[c for c in head_cols if c in summary] + um_cols +
                  [c for c in tail_cols if c in summary]]

# which volume column the plots and the strain table use
VOL_COL   = "V_disk_um3"  if "V_disk_um3"  in summary else "V_disk_px3"
TRUNC_COL = "V_trunc_um3" if "V_trunc_um3" in summary else "V_trunc_px3"
CAP_COL   = "V_cap_um3"   if "V_cap_um3"   in summary else "V_cap_px3"
BASE_COL  = ("V_base_um3" if "V_base_um3" in summary else "V_base_px3") \
            if BASELINE_Y_FULL else None
LEN_UNIT = "um" if SCALE_PX_PER_UM else "px"
VOL_UNIT = f"{LEN_UNIT}^3"
H_COL, A_COL = ("h_um", "a_um") if SCALE_PX_PER_UM else ("h_px", "a_px")

# --- is the calibration plausible for THIS set of images? ------------------------------------
# The crop offsets take care of themselves, but SCALE_PX_PER_UM does not: it belongs to one
# magnification, one working distance and one image size. Re-calibrate whenever any of those
# change, and never carry a scale across a resize - a half-size copy halves the true px/um.
if SCALE_PX_PER_UM:
    fw = int(summary["frame_w"].iloc[0])
    print(f"scale check: 1 px = {1/SCALE_PX_PER_UM:.3f} um  ->  the {fw}px-wide frame covers "
          f"{fw/SCALE_PX_PER_UM/1000:.1f} mm")
    print(f"             widest bead {2*summary['a_um'].max()/1000:.2f} mm across, "
          f"tallest {summary['h_um'].max()/1000:.2f} mm high")
    print("             if those do not match the real bead, SCALE_PX_PER_UM is wrong: volumes")
    print("             scale as its cube, though shrinkage and strain are unaffected by it.")
else:
    print("SCALE_PX_PER_UM is None -> volumes are in px^3. Shrinkage and strain are still valid,")
    print("since the scale cancels in a ratio; absolute volumes are not.")

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
summary.drop(columns=["path"])

## Step 3 - per-row `r_i` tables for every image

One long table holding every row of every bead, in both cropped and full-image coordinates
(`Y_full`, `x_left_full`, `x_right_full` are what you hover over in IrfanView on the **original**
photo). The same data also goes into one sheet per image in the workbook.

In [ ]:
profiles = pd.concat(
    [r["profile"].assign(index=r["metrics"]["index"],
                         label=r["metrics"]["label"],
                         image=r["metrics"]["image"])
     for r in results],
    ignore_index=True,
)
cols = ["index", "label", "image"] + [c for c in profiles.columns if c not in ("index", "label", "image")]
profiles = profiles[cols]

print(f"{len(profiles)} rows across {profiles['image'].nunique()} image(s)")
profiles.head(10)

### Save everything

`bead_volumes.xlsx` gets a `summary` sheet plus one sheet per image; the same content is written
as CSVs for anything that does not like Excel.

In [ ]:
def sheet_name(label, used):
    """Excel sheet names: <=31 chars, no []:*?/\\, and unique."""
    s = re.sub(r"[\[\]:*?/\\]", "_", str(label))[:31] or "sheet"
    base, k = s, 1
    while s in used:
        suffix = f"_{k}"
        s = base[:31 - len(suffix)] + suffix
        k += 1
    used.add(s)
    return s

xlsx = f"{OUT_PREFIX}.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
    summary.drop(columns=["path"]).to_excel(xl, sheet_name="summary", index=False)
    used = {"summary"}
    for r in results:
        r["profile"].to_excel(xl, sheet_name=sheet_name(r["metrics"]["label"], used), index=False)

summary.to_csv(f"{OUT_PREFIX}_summary.csv", index=False)
profiles.to_csv(f"{OUT_PREFIX}_profiles.csv", index=False)

print("saved:")
print(" ", xlsx, f"(summary + {len(results)} per-image sheet(s))")
print(" ", f"{OUT_PREFIX}_summary.csv")
print(" ", f"{OUT_PREFIX}_profiles.csv")

## Step 4 - plots

Colours: with up to 8 beads each one gets its own fixed colour; beyond that the beads are shaded
light-to-dark in processing order, since a long run is really a time sequence rather than 20
unrelated categories.

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]   # fixed categorical order, never cycled
INK, INK_SOFT, GRIDC = "#0b0b0b", "#52514e", "#d8d7d2"

from matplotlib.colors import LinearSegmentedColormap, Normalize
CMAP = LinearSegmentedColormap.from_list(   # viridis, ends trimmed so no bead is near-white
    "bead_seq", plt.get_cmap("viridis")(np.linspace(0.12, 0.92, 256)))

def colors_for(n):
    if n <= len(SERIES):
        return SERIES[:n]
    return [CMAP(v) for v in np.linspace(0, 1, n)]

def order_colorbar(fig, ax, n):
    """Colour key for sets too big to legend: colour = position in the processing order."""
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=Normalize(vmin=0, vmax=n - 1))
    cb = fig.colorbar(sm, ax=ax, pad=0.02)
    cb.set_label("image index (processing order)", color=INK_SOFT, fontsize=9)
    cb.ax.tick_params(colors=INK_SOFT, labelsize=8)
    cb.outline.set_visible(False)
    return cb

def tidy(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRIDC)
    ax.grid(True, axis=grid_axis, color=GRIDC, linewidth=0.8, alpha=0.9)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_SOFT, labelsize=9)
    if title:  ax.set_title(title, color=INK, fontsize=12, pad=10)
    if xlabel: ax.set_xlabel(xlabel, color=INK_SOFT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK_SOFT, fontsize=10)
    return ax

COLORS = colors_for(len(results))
LABELS = [r["metrics"]["label"] for r in results]
BIG_SET = len(results) > 8          # too many beads to legend / label individually
TIME_LABEL = f"time ({TIME_UNIT})" if TIME_REGEX else "image index"
print(f"{len(results)} bead(s); colours "
      + ("follow the processing order (light -> dark)" if BIG_SET else "one per image"))

### 4a - radius profiles `r(z)`, all beads on one axis

The shape of the whole bead, not just its volume. Beads that shrank uniformly stay geometrically
similar (same curve, smaller); a bead that only lost height, or that slumped outwards, shows up
here as a change in shape that a single volume number hides.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for res, c in zip(results, COLORS):
    p, m = res["profile"], res["metrics"]
    z = p["z_um"] if SCALE_PX_PER_UM else p["z_px"]
    r = p["r_i_um"] if SCALE_PX_PER_UM else p["r_i_px"]
    ax.plot(r, z, "-", color=c, linewidth=2, label=m["label"])
tidy(ax, "Bead radius profile r(z), apex at z = 0",
     f"r_i ({LEN_UNIT})", f"z below apex ({LEN_UNIT})", grid_axis="both")
ax.invert_yaxis()
if BIG_SET:
    order_colorbar(fig, ax, len(results))
else:
    ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT, title="image", title_fontsize=9)
plt.tight_layout(); plt.show()

### 4b - normalised shape, `r/a` vs `z/h`

The same profiles with the size divided out. Curves that lie on top of each other mean the beads
are the **same shape** at different sizes - which is what pure isotropic shrinkage looks like, and
what justifies reading a linear strain off the volume ratio. The dashed line is a perfect
spherical cap for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for res, c in zip(results, COLORS):
    p, m = res["profile"], res["metrics"]
    ax.plot(p["r_i_px"] / m["a_px"], p["z_px"] / m["h_px"], "-", color=c, linewidth=2,
            label=m["label"])

# reference spherical cap with the mean aspect ratio of the set
k = float(np.mean([r["metrics"]["aspect_h_over_a"] for r in results]))   # h/a
t = np.linspace(0, 1, 200)
R = (1 + k ** 2) / (2 * k)                    # sphere radius in units of a
ax.plot(np.sqrt(np.clip(R ** 2 - (R - k * t) ** 2, 0, None)), t,
        "--", color=INK_SOFT, linewidth=1.5, label=f"ideal cap (h/a={k:.2f})")

tidy(ax, "Normalised bead shape (size divided out)", "r / a", "z / h", grid_axis="both")
ax.invert_yaxis()
if BIG_SET:
    order_colorbar(fig, ax, len(results))
    ax.legend([ax.lines[-1]], [f"ideal cap (h/a={k:.2f})"], frameon=False, fontsize=9,
              labelcolor=INK_SOFT)
else:
    ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

### 4c - volume per image

Disk integration against the spherical-cap volume computed from the same `h` and `a`. The two
bars should be close; where they are not, that bead's silhouette is not a spherical cap and the
disk integration - which makes no shape assumption - is the number to trust.

In [ ]:
if len(summary) <= 12:
    x = np.arange(len(summary))
    w = 0.36                  # < half the 0.40 offset, so the paired bars keep a visible gap
    fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(summary) + 2), 5))
    b1 = ax.bar(x - 0.20, summary[VOL_COL], w, color=SERIES[0], label="disk integration")
    b2 = ax.bar(x + 0.20, summary[CAP_COL], w, color=SERIES[1], label="spherical cap (same h, a)")
    for bars in (b1, b2):
        ax.bar_label(bars, fmt="%.3g", fontsize=8, color=INK_SOFT, padding=2)
    ax.set_xticks(x, summary["label"], rotation=20, ha="right")
    tidy(ax, f"Bead volume per image ({VOL_UNIT})", None, f"V ({VOL_UNIT})")
else:
    # too many images for readable bars - same two series as lines against time
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(summary["time"], summary[VOL_COL], "-o", color=SERIES[0], linewidth=2,
            markersize=5, label="disk integration")
    ax.plot(summary["time"], summary[CAP_COL], "-o", color=SERIES[1], linewidth=2,
            markersize=5, label="spherical cap (same h, a)")
    tidy(ax, f"Bead volume per image ({VOL_UNIT})", TIME_LABEL, f"V ({VOL_UNIT})",
         grid_axis="both")
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

### 4d - shrinkage and strain

The point of the measurement. Taking image `REFERENCE_INDEX` as the un-shrunk state:

- **volumetric shrinkage** = `1 - V/V0`
- **linear strain** = `(V/V0)^(1/3) - 1`, the isotropic-shrinkage equivalent of that volume loss
  (negative = contraction). This is the number that pairs with a strain measured any other way,
  and it is only meaningful if 4b showed the shape staying similar.

Two panels rather than two y-axes on one plot, so neither curve's scale distorts the other.

Each panel carries **two** curves: the whole silhouette, and the same beads integrated only down to
their widest row. They answer the same question with and without the least trustworthy part of the
mask, so the gap between them is an honest error bar on the base. If they lie on top of each other,
the base is solid and the number is trustworthy; if they diverge, that difference is the real
uncertainty in your strain.

In [ ]:
if len(summary) < 2:
    print("Only one image - shrinkage needs at least two. Skipping.")
else:
    ref = summary.loc[summary["index"] == REFERENCE_INDEX]
    if ref.empty:
        ref = summary.iloc[[0]]
    t = summary["time"].values
    xlabel = TIME_LABEL

    series = [("whole silhouette", VOL_COL, SERIES[0])]
    if TRUNC_COL in summary:
        series.append(("truncated at the widest row", TRUNC_COL, SERIES[1]))
    if BASE_COL and BASE_COL in summary:
        series.append((f"filled down to the mat (Y={BASELINE_Y_FULL})", BASE_COL, SERIES[2]))

    fig, axes = plt.subplots(2, 1, figsize=(8, 8.5), sharex=True)
    for name, col, colour in series:
        V0 = float(ref[col].iloc[0])
        ratio = summary[col] / V0
        axes[0].plot(t, 100.0 * (1 - ratio), "-o", color=colour, linewidth=2, markersize=6,
                     label=name)
        axes[1].plot(t, 100.0 * (ratio ** (1 / 3) - 1), "-o", color=colour, linewidth=2,
                     markersize=6, label=name)

    tidy(axes[0], f"Volume shrinkage relative to '{ref['label'].iloc[0]}'", None,
         "1 - V/V0  (%)", grid_axis="both")
    tidy(axes[1], "Equivalent linear strain  (V/V0)^(1/3) - 1", xlabel,
         "linear strain (%)", grid_axis="both")
    for ax in axes:
        ax.axhline(0, color=GRIDC, linewidth=1)
        ax.margins(x=0.08, y=0.18)
        ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)

    # label the ends and a few points in between - one label per point is unreadable past ~8 beads
    V0 = float(ref[VOL_COL].iloc[0])
    lin_pct = 100.0 * ((summary[VOL_COL] / V0) ** (1 / 3) - 1)
    mark = sorted(set(np.linspace(0, len(t) - 1, min(len(t), 6)).round().astype(int)))
    for k in mark:
        axes[1].annotate(f"{lin_pct.iloc[k]:+.1f}%", (t[k], lin_pct.iloc[k]),
                         textcoords="offset points", xytext=(0, 10), ha="center",
                         fontsize=8, color=INK_SOFT)
    plt.tight_layout(); plt.show()

    if TRUNC_COL in summary:
        a = 100.0 * ((summary[VOL_COL].iloc[-1] / float(ref[VOL_COL].iloc[0])) ** (1/3) - 1)
        b = 100.0 * ((summary[TRUNC_COL].iloc[-1] / float(ref[TRUNC_COL].iloc[0])) ** (1/3) - 1)
        print(f"final linear strain: {a:+.2f}% (whole silhouette) vs {b:+.2f}% (truncated)")
        print("A gap between these two is the measurement's own uncertainty about the bead's base;")
        print("it is not something a longer run or more images will reduce.")

### 4e - height and base radius per image

A shrinking bead should lose height and base radius together. If only `h` falls while `a` holds,
the bead is pinned to the mat and shrinking anisotropically - the isotropic linear strain from 4d
then understates the vertical strain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, col, name in zip(axes, [H_COL, A_COL], ["height h", "base radius a"]):
    if len(summary) <= 12:
        bars = ax.bar(summary["label"], summary[col], color=SERIES[0], width=0.6)
        ax.bar_label(bars, fmt="%.1f", fontsize=8, color=INK_SOFT, padding=2)
        tidy(ax, f"{name} ({LEN_UNIT})", None, f"{name.split()[-1]} ({LEN_UNIT})")
        ax.tick_params(axis="x", labelrotation=20)
    else:
        ax.plot(summary["time"], summary[col], "-o", color=SERIES[0], linewidth=2, markersize=5)
        tidy(ax, f"{name} ({LEN_UNIT})", TIME_LABEL, f"{name.split()[-1]} ({LEN_UNIT})",
             grid_axis="both")
plt.tight_layout(); plt.show()

### 4f - how far each bead is from a spherical cap

`cap_diff_pct = 100 * (V_disk - V_cap) / V_cap`, one point per image, against a +/-2% guide band.
Near zero means the silhouette really is a spherical cap and the two methods agree. A large
*systematic* offset (every image on the same side) usually means `a` is being read off a flare at
the contact line rather than the true base - `A_METHOD = "robust"` is there for exactly that. A
single image jumping out of the band is a bad mask: go back to the Step 1b montage and look at it.

This replaces a 1:1 scatter, which is unreadable once the beads are all nearly the same size - the
axes zoom into a sliver and a 2% offset looks like a catastrophe.

In [ ]:
d = summary["cap_diff_pct"]

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axhspan(-2, 2, color=SERIES[0], alpha=0.07, zorder=0)
ax.axhline(0, color=INK_SOFT, linewidth=1.2, linestyle="--")
ax.plot(summary["time"], d, "-o", color=SERIES[0], linewidth=1.5, markersize=7,
        markeredgecolor="white", markeredgewidth=1.2)

# label only the images furthest from a cap - labelling all of them is a black smear
for i in d.abs().nlargest(min(3, len(d))).index:
    ax.annotate(summary.loc[i, "label"], (summary.loc[i, "time"], d[i]),
                textcoords="offset points", xytext=(0, 10), ha="center",
                fontsize=8, color=INK_SOFT)

ax.margins(x=0.06, y=0.25)
tidy(ax, "Departure from a spherical cap, per image", TIME_LABEL,
     "100 * (V_disk - V_cap) / V_cap  (%)", grid_axis="both")
ax.text(0.99, 0.04, "shaded band = +/-2%", transform=ax.transAxes, ha="right",
        fontsize=8, color=INK_SOFT)
plt.tight_layout(); plt.show()

print(f"mean departure {d.mean():+.2f}%  |  spread {d.std():.2f}%  |  "
      f"worst {d.abs().max():.2f}% ({summary.loc[d.abs().idxmax(), 'label']})")

### 4g - is the contact line staying put?

With the camera and the mat fixed, the row where the bead meets the mat is the same row in every
image. This plots, in **original image coordinates**, the apex, the widest row and the bottom of
each mask.

- the apex should drop as the bead loses height - that is the measurement working
- the widest row should be roughly flat. A drift means either the bead really is changing shape at
  its base (a bulging bead settling onto the mat has its widest point above the contact line, and
  that point moves down as it relaxes), or the shadow at the base is eating a different amount in
  each image
- the bottom of the mask wandering, especially while the widest row holds still, is the threshold
  losing the base - the same thing `base_taper` reports per image

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(summary["time"], summary["apex_Y_full"], "-o", color=SERIES[0], lw=2, ms=5, label="apex")
ax.plot(summary["time"], summary["Y_widest_full"], "-o", color=SERIES[1], lw=2, ms=5,
        label="widest row")
ax.plot(summary["time"], summary["Y_bottom_full"], "-o", color=SERIES[6], lw=2, ms=5,
        label="bottom of mask")
tidy(ax, "Key rows of each silhouette, in the original image", TIME_LABEL,
     "Y in the full image (px)", grid_axis="both")
ax.invert_yaxis()          # image rows count downwards
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

print(f"apex moves {summary['apex_Y_full'].max() - summary['apex_Y_full'].min():.0f} px, "
      f"widest row {summary['Y_widest_full'].max() - summary['Y_widest_full'].min():.0f} px, "
      f"mask bottom {summary['Y_bottom_full'].max() - summary['Y_bottom_full'].min():.0f} px")

## Step 5 - shrinkage / strain table

The same numbers as plot 4d in table form, appended to the workbook as a `shrinkage` sheet:

| column | meaning |
|---|---|
| `V_over_V0` | volume as a fraction of the reference bead |
| `shrinkage_pct` | `100 * (1 - V/V0)` - volume lost |
| `linear_strain_pct` | `100 * ((V/V0)^(1/3) - 1)` - isotropic linear strain, negative = contraction |
| `dV_pct_prev` | volume change relative to the **previous** image, for spotting a bad frame |

In [ ]:
ref_row = summary.loc[summary["index"] == REFERENCE_INDEX]
if ref_row.empty:
    ref_row = summary.iloc[[0]]
V0 = float(ref_row[VOL_COL].iloc[0])

cols = ["index", "label", "time", H_COL, A_COL, VOL_COL] + ([TRUNC_COL] if TRUNC_COL in summary else [])
shrink = summary[cols].copy()
shrink["V_over_V0"]         = shrink[VOL_COL] / V0
shrink["shrinkage_pct"]     = 100.0 * (1 - shrink["V_over_V0"])
shrink["linear_strain_pct"] = 100.0 * (shrink["V_over_V0"] ** (1 / 3) - 1)
shrink["dV_pct_prev"]       = 100.0 * shrink[VOL_COL].pct_change()
if TRUNC_COL in shrink:
    ratio_t = shrink[TRUNC_COL] / float(ref_row[TRUNC_COL].iloc[0])
    shrink["linear_strain_pct_trunc"] = 100.0 * (ratio_t ** (1 / 3) - 1)

with pd.ExcelWriter(xlsx, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
    shrink.to_excel(xl, sheet_name="shrinkage", index=False)
shrink.to_csv(f"{OUT_PREFIX}_shrinkage.csv", index=False)

print(f"reference = '{ref_row['label'].iloc[0]}',  V0 = {V0:.6g} {VOL_UNIT}")
print(f"saved shrinkage sheet to {xlsx} and {OUT_PREFIX}_shrinkage.csv")
shrink

## Notes / troubleshooting

- **Every image returns the same `h`, and it equals `y1 - y0` of the ROI** -> the crop is slicing
  the bead; the "apex" is the top edge of the crop. The volumes are then set by the ROI, not the
  bead, and the leftover variation between images is single-pixel threshold noise. Use Step 0b to
  get an ROI that encloses the whole bead and re-run. The batch now warns about this on its own.
- **`clipped` is non-empty for some images** -> same problem on those images only; widen the ROI,
  or give them their own `ROI_OVERRIDES` entry.
- **`h/a` above 1 (contact angle > 90 deg)** -> for a bead sitting on a mat this is usually a
  clipped crop rather than a real shape; check the montage.
- **A mask looks like a solid rectangle** -> flip `INVERT` and re-run from Step 1.
- **Text or frame caught in one mask** -> add that file to `ROI_OVERRIDES` with a tighter crop.
- **The bead's reflection in the mat is included** -> raise the ROI's bottom edge `y1` to the
  contact line; the integration stops at whatever `y1` you give it.
- **Volumes only mean something in µm^3 once `SCALE_PX_PER_UM` is set.** In pixels the *ratios*
  (and therefore the shrinkage and the strain) are still valid, since the scale cancels - so a
  strain analysis can be done without calibrating, an absolute volume cannot.
- **All images must share one scale and one working distance.** If the zoom changed between shots,
  the volumes are not comparable, and the shrinkage numbers are meaningless.
- `dz` is one pixel row, so the disk sum uses `dz = 1 px` and the volume comes out in px^3;
  dividing by `SCALE_PX_PER_UM^3` converts it, which is what the `_um3` columns do.